# Subsample datasets to 100 per year

## Housekeeping

In [1]:
# Libraries

import os
import pandas as pd
import dateutil
import re
import random
import shutil 
import numpy as np
from unidecode import unidecode
import importlib
import utils  
importlib.reload(utils)
from utils import * # If changing utils, must restart this file for changes to take effect

pd.options.mode.chained_assignment = None # suppress warnings when using slices to make new columns

In [9]:
# Set up directories

home = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu_Files/Paloma/complete_human/"

references = "C:/Users/maksiaevai.NCBI_NT/OneDrive - National Institutes of Health/Documents/Virus_Evolution/Avian_Flu/references/"

os.chdir(references)
human_variants = pd.read_csv("human_variants.csv")
# states_ref = pd.read_csv("states_ref.csv")

## Upload FASTAs

In [3]:
# Organize fastas

fastas = {}
for dirpath, dirs, files in os.walk(home + "to_subsample/"):
    for file in files:
        file_name = os.path.join(dirpath, file) # .split("/")[-1]
        if ".fasta" in file_name:
            fasta = df_from_fasta(file_name)
            fastas[file_name.split("/")[-1]] = fasta
    break # Do not go into subfolders

In [4]:
print(fastas)

{'europe_human+swine_pdmOnly_2009-2026_aln_trim_downsampled.fasta':                                             full_header  \
0     >MZ945846|A/swine/Spain/45690-9/2018|H1N2|Spai...   
1     >MW848689|A/swine/Spain/31001-2/2019|H1N1|Spai...   
2     >PP331791|A/swine/Spain/05165-1/2020|H1N1|Spai...   
3     >PP338521|A/swine/Spain/44593-1/2021|H1N1|Spai...   
4     >PQ107580|A/swine/Spain/44394-1/2021|H1N1|Spai...   
...                                                 ...   
8671  >EPI_ISL_20347483|A/England/01898391/2026|H1N1...   
8672  >EPI_ISL_20314783|A/Netherlands/10010/2026|H1N...   
8673  >EPI_ISL_20314781|A/Netherlands/10009/2026|H1N...   
8674  >EPI_ISL_20314766|A/Netherlands/10001/2026|H1N...   
8675  >EPI_ISL_20314765|A/Netherlands/10000/2026|H1N...   

                                               sequence  
0     atgaaggcaatactagtagttctgctatatacatttgcgaccacaa...  
1     atgaaggcaataatggtagttctgctatatacatttgcaaccgcag...  
2     atgaaggcaatactagtagttctgctgtatacatttacaaccg

## Subsample

In [5]:
# Create a dictionary of dictionaries of dataframes grouped by year

fastas_grouped = {} # Overall dictionary -- length is number of datasets needed

for key in fastas:
    years = {} # For each dataset, there are a number of years
    fasta = fastas[key]
    fasta["Year"] = fasta["full_header"].apply(lambda x: dateutil.parser.parse(x.split("|")[4]).year) # Find year
    fasta["Isolate"] = fasta["full_header"].apply(lambda x: x.split("|")[1].lower()) # if "human" in x else x.split("/")[3] if "/" in x else "unknown") #  if len(x.split("/")) > 2 else "unknown")
    fasta = fasta.drop_duplicates(subset="Isolate", keep="first")
    for year, rows in fasta.groupby("Year"): # Separate dataframe into multiple dataframes by year
        years[year] = rows # For each year, there are a number of entries that have that year
    fastas_grouped[key] = years


# Remove poor quality sequences
def calculate_ns(sequence, ref_length):
    allowable_characters = ["A", "T", "G", "C", "a", "t", "g", "c", "-"] # Anything else is N or other?
    other_characters = 0
    for character in sequence:
        if character not in allowable_characters:
            other_characters += 1
    proportion_ns = other_characters/ref_length
    return proportion_ns

hd_dfs = defaultdict(list)
# lq_dfs = defaultdict(list)
for file_fasta in fastas_grouped.keys():
    for year in fastas_grouped[file_fasta]: 
        year_fasta = fastas_grouped[file_fasta][year]
        # print(year_fasta)
        year_fasta["reference_sequence"] = year_fasta["sequence"].apply(lambda x: x.replace("-", ""))
        year_fasta["length"] = year_fasta["reference_sequence"].apply(len)
        year_fasta["proportion_ns"] = year_fasta["reference_sequence"].apply(lambda x: calculate_ns(x, year_fasta[year_fasta["reference_sequence"] == x]["length"].values[0]))
        average_len_thresh = year_fasta["length"].mean() # * 0.8
        hd_df = year_fasta[(year_fasta["proportion_ns"] <= 0.05) & (year_fasta["length"] >= average_len_thresh)]
        # print(hd_df)
        # lq_df = year_fasta[(year_fasta["proportion_ns"] > 0.05) | (year_fasta["length"] < average_len_thresh)]
        # print(lq_df)
        hd_dfs[file_fasta].append(hd_df)
        # lq_dfs[year_fasta].append(lq_df)

print(fastas_grouped)
print(hd_dfs)

{'europe_human+swine_pdmOnly_2009-2026_aln_trim_downsampled.fasta': {2009:                                             full_header  \
12    >EPI_ISL_73603|A/swine/Italy/85437/2009|H1N1|I...   
105   >EPI_ISL_195175|A/swine/England/1353/2009|H1N1...   
119   >EPI_ISL_129507|A/swine/England/MD0040352R/200...   
292   >EPI_ISL_103101|A/swine/Norway/02_11342/2009|H...   
339   >EPI_ISL_66077|A/swine/Italy/290271/2009|H1N1|...   
...                                                 ...   
4873  >EPI_ISL_60232|A/Estonia/2/2009_BT|H1N1|Estoni...   
4880  >EPI_ISL_93449|A/Paris/7973/2010|H1N1|Paris|20...   
4881  >EPI_ISL_93451|A/Paris/7975/2010|H1N1|Paris|20...   
4882  >EPI_ISL_60228|A/Latvia/6-1221/2009v|H1N1|Latv...   
4884  >EPI_ISL_93450|A/Paris/7974/2010|H1N1|Paris|20...   

                                               sequence  Year  \
12    atgaaggcaatactagtagttctgctatatacatttgcaaccgcaa...  2009   
105   atgaaggcaatactaatagttctgctatatacatttgcaaccgcaa...  2009   
119   atgaaggcaatacta

In [34]:
subsampled_dfs = defaultdict(list) # Overall subsampled dictionary -- length is number of datasets needed
# for key in fastas_grouped:
#     dataset = fastas_grouped[key] # Dataset
#     df = pd.DataFrame() # Hold subsampled data
#     for year_key in dataset: # Dictionary of years and their dataframes 
#         year_df = dataset[year_key] # One year and its data
#         # If there are more than 100 entries, subsample a random 100 
#         subsampled = year_df[["full_header", "sequence"]].sample(n=100, random_state=2008) if len(year_df) > 100 else year_df[["full_header", "sequence"]]
#         # print(subsampled)
#         df = pd.concat([df, subsampled]) # Add subsampled data to dataframe
#     subsampled_dfs[key] = df # Add dataframe to dictionary of datasets

print(human_variants)
variant_list = human_variants

# print(subsampled_dfs)

for file_list in hd_dfs:
    df = pd.DataFrame() # Hold subsampled data
    for year_fasta in hd_dfs[file_list]:
        # print(year_fasta)
        year = year_fasta["Year"].values[0] # All rows are the same year
        year_fasta_variants = year_fasta[year_fasta["full_header"].isin(variant_list["variants"].values)][["full_header", "sequence"]]
        year_fasta_paloma = year_fasta[year_fasta["full_header"].str.contains("Iberian|White|Other|wild_boar")][["full_header", "sequence"]]
        # If there are more than 100 entries, subsample a random 100 
        subsampled = year_fasta[~year_fasta["full_header"].isin(pd.concat([year_fasta_variants, year_fasta_paloma]))][["full_header", "sequence"]].sample(n=100, random_state=2008) if len(year_fasta) > 100 else year_fasta[["full_header", "sequence"]]
        df = pd.concat([df, year_fasta_variants, year_fasta_paloma, subsampled]) # Add subsampled data to dataframe
        df["accession"] = df["full_header"].apply(lambda x: x.split("|")[0])
        df = df.drop_duplicates(subset="accession", keep="first")
        df = df[["full_header", "sequence"]]
    subsampled_dfs[file_list].append(df) # Add dataframe to dictionary of datasets

print(subsampled_dfs)

                           variants
0            A/Switzerland/114/2023
1               A/Navarra/4050/2022
2    A/Catalonia/NSAV198309324/2024
3    A/Catalonia/NSAV600497186/2026
4    A/Catalonia/NSAV198289092/2023
5      A/Nordrhein-Westfalen/8/2022
6                  A/Hessen/47/2020
7   A/Mecklenburg-Vorpommern/1/2021
8        A/Netherlands/10370-2/2020
9             A/Bretagne/24241/2021
10         A/Netherlands/11748/2022
11      A/Netherlands/Gent-193/2019
12                  A/Pavia/65/2016
13          A/Netherlands/3315/2016
14         A/England/234600203/2023
15           A/Austria/1445532/2021
16         A/Netherlands/10534/2023
17         A/Greece/SRIHER-001/2022
18               A/Verona/2810/2009
defaultdict(<class 'list'>, {'europe_human+swine_pdmOnly_2009-2026_aln_trim_downsampled.fasta': [                                            full_header  \
12    >EPI_ISL_73603|A/swine/Italy/85437/2009|H1N1|I...   
105   >EPI_ISL_195175|A/swine/England/1353/2009|H1N1...   
119   

## Download FASTAs

In [35]:
# # Prepare for download
# os.chdir(home + "to_subsample/")
for key in subsampled_dfs:
    file_name = "subsampled_" + key # Create file name
    fasta = subsampled_dfs[key][0].reset_index()
    print(fasta)
    df_to_fasta(fasta, file_name, home + "paloma_to_concat/")


      index                                        full_header  \
0        12  >EPI_ISL_73603|A/swine/Italy/85437/2009|H1N1|I...   
1       105  >EPI_ISL_195175|A/swine/England/1353/2009|H1N1...   
2       119  >EPI_ISL_129507|A/swine/England/MD0040352R/200...   
3       292  >EPI_ISL_103101|A/swine/Norway/02_11342/2009|H...   
4       339  >EPI_ISL_66077|A/swine/Italy/290271/2009|H1N1|...   
...     ...                                                ...   
2156   8641  >EPI_ISL_20347449|A/Badajoz/18792682/2026|H1N1...   
2157   8416  >EPI_ISL_20335093|A/Netherlands/00060/2026|H1N...   
2158   8405  >EPI_ISL_20331737|A/England/01893260/2026|H1N1...   
2159   8441  >EPI_ISL_20336166|A/Catalonia/NSGT198345404/20...   
2160   8313  >EPI_ISL_20349499|A/England/01902646/2026|H1N1...   

                                               sequence  
0     atgaaggcaatactagtagttctgctatatacatttgcaaccgcaa...  
1     atgaaggcaatactaatagttctgctatatacatttgcaaccgcaa...  
2     atgaaggcaatactaatagttctgcta

## Build datasets

In [37]:
os.chdir(home + "paloma_to_concat/")
paloma_fasta = df_from_fasta("H1_Paloma_wLineage_wDatesAdded_v8.fasta")

full_subsampled_dfs = {}
for key in subsampled_dfs:
    full_df = pd.DataFrame()
    for f in subsampled_dfs[key]:
        fasta_df = f
        fasta_df["Accession"] = fasta_df["full_header"].apply(lambda x: x.split("|")[0].replace(">", ""))
        fasta_df["Host"] = fasta_df["full_header"].apply(lambda x: "human" if "human" in x else x.split("/")[1] if "/" in x else "unknown") #  if len(x.split("/")) > 1 else "unknown")
        fasta_df["Isolate"] = fasta_df["full_header"].apply(lambda x: x.split("/")[2] if "human" in x else x.split("/")[3] if "/" in x else "unknown") #  if len(x.split("/")) > 2 else "unknown")
        fasta_df["Subtype"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-4])
        fasta_df["Geo_Location"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-3])
        fasta_df["Collection_Date"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-2])
        fasta_df["Host_Type"] = fasta_df["full_header"].apply(lambda x: x.split("|")[-1])
        
        print(fasta_df)
        full_df = pd.concat([full_df, fasta_df])
    full_subsampled_dfs[key] = full_df

print(full_subsampled_dfs)

                                            full_header  \
12    >EPI_ISL_73603|A/swine/Italy/85437/2009|H1N1|I...   
105   >EPI_ISL_195175|A/swine/England/1353/2009|H1N1...   
119   >EPI_ISL_129507|A/swine/England/MD0040352R/200...   
292   >EPI_ISL_103101|A/swine/Norway/02_11342/2009|H...   
339   >EPI_ISL_66077|A/swine/Italy/290271/2009|H1N1|...   
...                                                 ...   
8641  >EPI_ISL_20347449|A/Badajoz/18792682/2026|H1N1...   
8416  >EPI_ISL_20335093|A/Netherlands/00060/2026|H1N...   
8405  >EPI_ISL_20331737|A/England/01893260/2026|H1N1...   
8441  >EPI_ISL_20336166|A/Catalonia/NSGT198345404/20...   
8313  >EPI_ISL_20349499|A/England/01902646/2026|H1N1...   

                                               sequence         Accession  \
12    atgaaggcaatactagtagttctgctatatacatttgcaaccgcaa...     EPI_ISL_73603   
105   atgaaggcaatactaatagttctgctatatacatttgcaaccgcaa...    EPI_ISL_195175   
119   atgaaggcaatactaatagttctgctatatacatttgcaaccgcaa...    E

In [40]:
# Seasonal H1

seasonal_fasta = pd.concat([paloma_fasta[paloma_fasta["full_header"].str.contains("seasonal-like")], full_subsampled_dfs["human_euro_seasonal_h1n1_01-01-1977--12-31-2008.fasta"]]) # .str.contains("2000s-like") [paloma_fasta["full_header"]].str.contains("2000s-like")
seasonal_fasta = seasonal_fasta[~seasonal_fasta["sequence"].str.contains("p")]
print(seasonal_fasta)

df_to_fasta(seasonal_fasta.reset_index(), "H1_paloma_human_euro_01-01-1977--12-31-2008.fasta", home + "paloma_to_concat/")

                                           full_header  \
6    >MF872818|A/swine/Spain/45690-2/2016|H1N1|Spai...   
7    >MZ945809|A/swine/Spain/45600-1/2017|H1N2|Spai...   
8    >MZ945740|A/swine/Spain/45600-2/2017|H1N2|Spai...   
9    >MZ945748|A/swine/Spain/45600-3/2017|H1N2|Spai...   
10   >MZ945878|A/swine/Spain/45600-4/2017|H1N2|Spai...   
..                                                 ...   
18   >EPI_ISL_164020|A/Paris/1164/2008|H1N1|Paris|2...   
295  >EPI_ISL_23310|A/Kaliningrad/11/2008|H1N1|Kali...   
286  >EPI_ISL_23322|A/St_Petersburg/85/2008|H1N1|St...   
661  >EPI_ISL_157235|A/Dnipropetrovsk/117/2008|H1N1...   
200  >EPI_ISL_27695|A/Denmark/122/2008|H1N1|Denmark...   

                                              sequence       Accession   Host  \
6    atgaaagctaaactactaatcctgttatgtacactttcagccacaa...             NaN    NaN   
7    atgaaagctaaactactaatcctgttatgtacactttcagccacag...             NaN    NaN   
8    atgaaagctaaactactaatcctgttatgtacactttcagccacag...      